In [0]:
# Define widgets with default values
dbutils.widgets.text("quote_id", "CR9999")
dbutils.widgets.text("catalog", "lrcatalog")
dbutils.widgets.text("schema", "agentic_underwriting")

#%pip install -U databricks-agents databricks-openai databricks-langchain mlflow
#dbutils.library.restartPython()


In [0]:
# Get widget values
quote_id = dbutils.widgets.get("quote_id")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

### Here we start with tool definitions for our agent.


### 🔍 Function: `make_get_quote_details`

This function factory returns a callable that retrieves details of a specific motor insurance quote from the `quotes` table in the specified catalog and schema.

- **Input:** `quote_id` (case-insensitive, trimmed string)
- **Output:** Markdown-formatted table with quote details, or an error message if not found
- **Use case:** Can be plugged into LangChain agents or UI apps to display quote details based on user input

In [0]:
def make_get_quote_details(catalog, schema):
    def _fn(quote_id: str) -> str:
        quote_id_clean = quote_id.strip()
        df = spark.sql(f"""
            SELECT * FROM {catalog}.{schema}.commercial_quotes
            WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
        """).toPandas()

        if df.empty:
            return f"🚫 Quote not found for quote_id: {quote_id_clean}"
        
        markdown = df.to_markdown(index=False)
        return f"✅ Quote found:\n\n{markdown}"
    return _fn


### 🔍 Function: `make_validate_claims`

This function factory creates a callable that validates disclosed claims for a customer based on their full name and postcode.

- **Input:** A string in the format `'First Last, POSTCODE'`
- **Output:** A Markdown-formatted table of matching claim validation records, or an error message
- **Use case:** Useful in LangChain agents or UI flows to check if the customer’s disclosed claims match stored records before underwriting

In [0]:
def make_validate_claims(catalog, schema):
    def _fn(quote_id: str) -> str:
        qid = quote_id.strip()

        # Get the commercial quote row
        quote_df = spark.sql(f"""
            SELECT quote_id, company_name, postcode, previous_claims
            FROM {catalog}.{schema}.commercial_quotes
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """).toPandas()

        if quote_df.empty:
            return f"🚫 Quote not found in commercial_quotes for quote_id: {qid}"

        qrow = quote_df.iloc[0]
        company_name, postcode, declared_claims = qrow["company_name"], qrow["postcode"], qrow["previous_claims"]

        # Lookup disclosure validation directly by quote_id
        val_df = spark.sql(f"""
            SELECT quote_id, claims_amount AS actual_claims, ncd_amount
            FROM {catalog}.{schema}.global_claims_disclosure_validation
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """).toPandas()

        if val_df.empty:
            return f"⚠️ No disclosure validation record found for quote_id {qid} ({company_name}, {postcode})."

        actual_claims = val_df.iloc[0]["actual_claims"]
        ncd_amount   = val_df.iloc[0]["ncd_amount"]

        status = "✅ Match" if declared_claims == actual_claims else "❌ Mismatch"

        summary = pd.DataFrame([{
            "quote_id": qid,
            "company_name": company_name,
            "postcode": postcode,
            "declared_claims": declared_claims,
            "actual_claims": actual_claims,
            "ncd_amount": ncd_amount,
            "validation_status": status
        }]).to_markdown(index=False)

        return f"🔍 Claims disclosure validation:\n\n{summary}"
    
    return _fn


### 🔍 Function: `make_get_call_transcript`

This function factory returns a callable that retrieves the call transcript associated with a specific motor insurance quote from the `sales_call_transcripts` table.

- **Input:** `quote_id` (string)
- **Output:** Raw text of the call transcript, or a message if no transcript is found
- **Use case:** Can be used in LangChain agents or apps to provide customer interaction history for underwriting or sales analysis

In [0]:
def make_get_call_transcript(catalog, schema):
    def _fn(quote_id: str) -> str:
        quote_id_clean = quote_id.strip()
        
        df = spark.sql(f"""
            SELECT quote_id, call_transcript
            FROM {catalog}.{schema}.global_sales_call_transcripts
            WHERE LOWER(quote_id) = LOWER('{quote_id_clean}')
        """).toPandas()
        
        if df.empty:
            return f"🚫 No call transcript found for quote_id: {quote_id_clean}"
        
        row = df.iloc[0]
        transcript = (row["call_transcript"] or "").strip()
        if not transcript:
            return f"⚠️ Transcript record exists but is empty for quote_id: {quote_id_clean}"
        
        # Basic readability stats
        lines = [ln for ln in transcript.splitlines() if ln.strip()]
        num_lines = len(lines)
        num_chars = len(transcript)
        
        # Return nicely formatted markdown
        header = f"📞 Call transcript for **{row['quote_id']}**  \nLines: {num_lines} · Characters: {num_chars}"
        body = f"\n```\n{transcript}\n```"
        return f"{header}\n{body}"
    return _fn


### 🔍 Function: `score_quote_tool`

This function scores a motor insurance quote using a basic risk model based on age, vehicle type, number of claims, and no claims discount (NCD).

- **Input:** A dictionary or JSON string with keys: `age`, `vehicle_type`, `ncd_declared`, and `claims_declared`
- **Output:** A dictionary containing the calculated `price`, the `model_name`, and optionally an `error` message
- **Use case:** Can be used in LangChain agents or quote evaluation tools to simulate a pricing model based on risk factors

In [0]:
import pandas as pd

def make_score_commercial_quote(catalog, schema):
    def _fn(quote_id: str) -> str:
        qid = quote_id.strip()

        # 1) Pull the quote
        qdf = spark.sql(f"""
            SELECT quote_id, company_name, industry, line_of_business, postcode,
                   turnover, num_employees, sum_insured, deductible, previous_claims, channel
            FROM {catalog}.{schema}.commercial_quotes
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """).toPandas()
        if qdf.empty:
            return f"🚫 Quote not found in {catalog}.{schema}.commercial_quotes for quote_id: {qid}"
        q = qdf.iloc[0]

        # 2) Industry × turnover risk (join by industry AND turnover in band)
        idf = spark.sql(f"""
            SELECT industry, turnover_band, turnover_min, turnover_max,
                   risk_segment, base_rate_multiplier
            FROM {catalog}.{schema}.commercial_industry_risk_profile
            WHERE LOWER(industry) = LOWER('{q['industry']}')
              AND {int(q['turnover'])} >= turnover_min
              AND {int(q['turnover'])} <  turnover_max
        """).toPandas()

        if idf.empty:
            return f"⚠️ No industry risk mapping for {q['industry']} at turnover £{q['turnover']:,}."

        irisk = idf.iloc[0]
        base_rate_mult = float(irisk["base_rate_multiplier"])

        # 3) Property protection & territory risk (by postcode)
        pdf = spark.sql(f"""
            SELECT postcode, has_sprinklers, alarm_grade, fire_resistance_class, territory_risk_level
            FROM {catalog}.{schema}.commercial_property_protection
            WHERE LOWER(postcode) = LOWER('{q['postcode']}')
        """).toPandas()

        has_sprinklers = False
        alarm_grade = "None"
        fire_res = "Low"
        territory = "mid_low"
        if not pdf.empty:
            prow = pdf.iloc[0]
            has_sprinklers = bool(prow["has_sprinklers"])
            alarm_grade = (prow["alarm_grade"] or "None")
            fire_res = (prow["fire_resistance_class"] or "Low")
            territory = (prow["territory_risk_level"] or "mid_low")

        # Territory factor
        terr_factor_map = {"high": 1.15, "mid_high": 1.08, "mid_low": 1.03, "low": 1.00}
        territory_factor = float(terr_factor_map.get(str(territory), 1.03))

        # Protection factor
        alarm_factor_map = {"None": 1.05, "Bells Only": 1.02, "Monitored": 0.99, "Police Response": 0.97}
        fire_res_map = {"Low": 1.05, "Medium": 1.00, "High": 0.97}

        protection_factor = (0.98 if has_sprinklers else 1.00) \
                            * float(alarm_factor_map.get(alarm_grade, 1.02)) \
                            * float(fire_res_map.get(fire_res, 1.00))

        # 4) Liability exposure bands (by num_employees)
        ldf = spark.sql(f"""
            SELECT employees_min, employees_max, rate_per_employee
            FROM {catalog}.{schema}.commercial_liability_exposure_bands
            WHERE {int(q['num_employees'])} >= employees_min
              AND {int(q['num_employees'])} <= employees_max
        """).toPandas()
        rate_per_emp = float(ldf.iloc[0]["rate_per_employee"]) if not ldf.empty else 40.0

        # 5) Components & premium
        sum_insured = float(q["sum_insured"])
        num_emp = int(q["num_employees"])
        deductible = float(q["deductible"])
        prev_claims = int(q["previous_claims"])
        lob = str(q["line_of_business"])

        base_property = 250.0 + (sum_insured * 0.001)
        liability_part = (rate_per_emp * num_emp) if lob.lower() in ("liability", "combined") else 0.0
        claims_loading = prev_claims * 500.0
        deductible_credit = -min(deductible * 0.05, 500.0)

        multiplier = base_rate_mult * territory_factor * protection_factor
        premium = (base_property + liability_part + claims_loading + deductible_credit) * multiplier

        # 6) Markdown output
        inputs_md = pd.DataFrame([{
            "quote_id": qid,
            "company": q["company_name"],
            "industry": q["industry"],
            "LOB": lob,
            "postcode": q["postcode"],
            "turnover": f"£{q['turnover']:,}",
            "employees": num_emp,
            "sum_insured": f"£{int(sum_insured):,}",
            "deductible": f"£{int(deductible):,}",
            "previous_claims": prev_claims
        }]).to_markdown(index=False)

        factors_md = pd.DataFrame([{
            "base_property": round(base_property, 2),
            "liability_part": round(liability_part, 2),
            "claims_loading": round(claims_loading, 2),
            "deductible_credit": round(deductible_credit, 2),
            "base_rate_mult": round(base_rate_mult, 3),
            "territory_factor": round(territory_factor, 3),
            "protection_factor": round(protection_factor, 3),
            "TOTAL_multiplier": round(multiplier, 3),
            "premium": round(premium, 2)
        }]).to_markdown(index=False)

        extras_md = pd.DataFrame([{
            "has_sprinklers": has_sprinklers,
            "alarm_grade": alarm_grade,
            "fire_resistance_class": fire_res,
            "territory_risk_level": territory,
            "rate_per_employee": rate_per_emp
        }]).to_markdown(index=False)

        return (
            "🧮 **Commercial quote scoring**\n\n"
            "#### Inputs\n" + inputs_md + "\n\n"
            "#### Factors & Premium\n" + factors_md + "\n\n"
            "#### Protection & Exposure Details\n" + extras_md
        )
    return _fn


### 🏡 Function: `validate_property_attributes`

This function retrieves property-level risk data for a given postcode, such as garage and driveway availability and overall property risk level.

- **Input:** A UK postcode as a plain string (e.g., `'CR3 6JE'`)
- **Output:** A markdown table of the matching row from the `property_attributes` table, or a message if no match is found
- **Use case:** Used in LangChain agents or underwriting pipelines to enrich or validate property context based on location

In [0]:
import pandas as pd

def make_validate_property_risk(catalog, schema):
    def _fn(quote_id: str) -> str:
        qid = quote_id.strip()

        # 1) Get the quote’s postcode
        qdf = spark.sql(f"""
            SELECT quote_id, company_name, postcode
            FROM {catalog}.{schema}.commercial_quotes
            WHERE LOWER(quote_id) = LOWER('{qid}')
        """).toPandas()

        if qdf.empty:
            return f"🚫 Quote not found for quote_id: {qid}"

        q = qdf.iloc[0]
        pc = q["postcode"]

        # 2) Get property protection features for that postcode
        pdf = spark.sql(f"""
            SELECT postcode, has_sprinklers, alarm_grade, fire_resistance_class, territory_risk_level
            FROM {catalog}.{schema}.commercial_property_protection
            WHERE LOWER(postcode) = LOWER('{pc}')
        """).toPandas()

        if pdf.empty:
            return f"⚠️ No property risk features found for postcode {pc}."

        prow = pdf.iloc[0]

        # 3) Simple risk validations
        flags = []
        if not prow["has_sprinklers"]:
            flags.append("❌ No sprinklers installed (higher fire risk)")
        if str(prow["alarm_grade"]).lower() in ("none", "nan"):
            flags.append("❌ No alarm system declared")
        if prow["fire_resistance_class"] == "Low":
            flags.append("⚠️ Low fire resistance construction")
        if prow["territory_risk_level"] == "high":
            flags.append("⚠️ Located in high-risk territory")

        status = "✅ All key protection features acceptable" if not flags else " / ".join(flags)

        # 4) Markdown summary
        features_md = pd.DataFrame([{
            "quote_id": qid,
            "company_name": q["company_name"],
            "postcode": pc,
            "has_sprinklers": prow["has_sprinklers"],
            "alarm_grade": prow["alarm_grade"],
            "fire_resistance_class": prow["fire_resistance_class"],
            "territory_risk_level": prow["territory_risk_level"],
            "validation_notes": status
        }]).to_markdown(index=False)

        return f"🏠 Property risk features validation:\n\n{features_md}"
    
    return _fn

In [0]:
# def make_get_driver_risk(catalog, schema):
#     def _fn(age_str: str) -> str:
#         def escape_sql(v): return v.replace("'", "''")
#         try:
#             age = int(age_str)
#         except ValueError:
#             return "Please provide a valid age as a number."

#         df = spark.sql(f"""
#             SELECT * FROM {catalog}.{schema}.driver_risk_profile
#             WHERE age = {age}
#         """).toPandas()

#         if df.empty:
#             return f"No risk profile found for age {age}."
#         return df.to_markdown(index=False)
#     return _fn


### 🧰 Tool Definitions for LangChain Agent

This section defines the set of tools available to the LangChain agent, each wrapping a callable function with a clear name and description.

- **Tools included:**
  - `Get Quote Details`: Returns declared customer and quote info by quote ID
  - `Validate Claims`: Returns verified NCD and claims using name and postcode
  - `Get Call Transcript`: Retrieves the sales call transcript for a quote ID
  - `Score Quote`: Calculates a new quote price using risk-based inputs

- **Use case:** These tools enable the agent to retrieve and cross-check underwriting data, simulate new pricing, and analyze sales interactions in real-time

In [0]:
from langchain.tools import Tool

tools = [
    Tool.from_function(
        make_get_quote_details(catalog, schema),
        name="Get Quote Details",
        description=(
            "Returns the declared commercial quote data including company name, industry, "
            "line of business, postcode, turnover, number of employees, sum insured, "
            "deductible, and declared previous claims. "
            "Input: provide the quote ID (e.g. 'CR9999')."
        )
    ),
    Tool.from_function(
        make_validate_claims(catalog, schema),
        name="Validate Claims",
        description=(
            "Validates the declared claims against disclosure records. "
            "It checks first name, last name, and postcode of the company contact against "
            "global_claims_disclosure_validation. "
            "Returns declared vs actual claims, NCD (if relevant), and validation status. "
            "Input: provide the quote ID."
        )
    ),
    Tool.from_function(
        make_get_call_transcript(catalog, schema),
        name="Get Call Transcript",
        description=(
            "Returns the sales call transcript for a given commercial quote ID, if available. "
            "Use this to analyse what was said by the customer vs what was declared in the quote "
            "and validation data. Look for potential inconsistencies or errors. "
            "Input: provide the quote ID."
        )
    ),
    Tool.from_function(
        make_score_commercial_quote(catalog, schema),
        name="Score Quote",
        description=(
            "Scores a commercial quote using enrichment tables (industry risk, property protection, "
            "liability exposure). Returns a calculated premium along with a breakdown of factors and "
            "multipliers used. "
            "Input: provide the quote ID."
        )
    ),
    Tool.from_function(
        make_validate_property_risk(catalog, schema),
        name="Validate Property Risk Features",
        description=(
            "Returns postcode-level property risk features for a commercial quote, such as sprinklers, "
            "alarm grade, fire resistance class, and territory risk level. Includes validation notes "
            "highlighting any risk concerns (e.g., no sprinklers, high-risk territory). "
            "Input: provide the quote ID."
        )
    )
]


### 🧠 Agent Initialization

This cell initializes a LangChain agent using a Databricks-hosted LLM (`meta-llama-3-1-70b-instruct`) and a set of predefined tools.

- **LLM:** `ChatDatabricks` connected to a specified endpoint
- **Agent type:** `ZERO_SHOT_REACT_DESCRIPTION` — allows reasoning over tool descriptions without examples
- **Settings:**
  - `verbose=True` for step-by-step logging
  - `handle_parsing_errors=True` to gracefully manage response formatting issues

- **Use case:** Powers a dynamic, tool-using agent capable of answering insurance-related questions or performing quote evaluations

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import initialize_agent, AgentType

llm = ChatDatabricks(endpoint="databricks-meta-llama-3-3-70b-instruct")

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True
)


### 📝 Agent Execution Workflow

This cell runs a multi-step underwriting evaluation process through the LangChain agent using the specified quote ID.

- **Goal:** Assist an underwriter in checking quote accuracy and flagging issues before approval
- **Steps:**
  1. **Get Quote Details** – fetch declared quote information
  2. **Validate Claims** – check against verified claims/NCD data
  3. **Get Call Transcript** – verify customer disclosures against quote data
  4. **Identify Errors** – if call center input error is suspected, rescore using `Score Quote`
  5. **Summarize** – return a comprehensive decision message

- **Use case:** Enables end-to-end decision support with data validation, transcript analysis, and dynamic pricing review

In [0]:
agent_output = agent.run(f"""
You are an underwriter for COMMERCIAL insurance policies. You will check a quote and decide whether it should be approved or reviewed.

Step 1: Use the 'Get Quote Details' tool with quote ID {quote_id}. If the quote is not found, respond with a message and stop the process. Otherwise, return the quote details (company name, industry, line of business, postcode, turnover, employees, sum insured, deductible, previous claims).

Step 2: Use the 'Validate Claims'. If the customer/contact does not exist in validation data for the matching first name, last name and postcode, flag potential fraud or missing disclosure data, respond with the correct message and STOP the process. Otherwise compare declared previous_claims from the quote with the validated actual_claims. Decide whether this passes or requires review and return the relevant message.

Step 3: Use the 'Get Call Transcript' tool to retrieve the sales call for this quote. Check if values stated in the call (e.g., company name, postcode, turnover, employees, claims) match the quote. If the customer provided correct details in the call but the quote shows different values, flag a possible data-entry error by the call handler.

Step 4: Use the 'Validate Property Risk Features' tool to retrieve postcode-level features (sprinklers, alarm grade, fire resistance class, territory risk level). Compare these features against any assumptions used in the quote (if applicable). If there are mismatches that materially impact risk (e.g., no sprinklers where assumed, high-risk territory), flag for review and record the differences. If nothing material is found, proceed.

Step 5: Use 'Score Quote' to compute the commercial premium using enrichment tables (industry × turnover band, protection, liability exposure). If earlier steps suggest incorrect assumptions or data-entry issues, clearly note them and recommend human review; the scoring result here should be treated as a baseline computed from currently stored data.

Step 6: Summarize the above steps in bullet points, listing which tools were used and their outputs. Include key fields (industry, LOB, turnover, employees, protection features, territory risk, claims validation result) and the premium with factor breakdown. State clearly whether the quote should be APPROVED or REFERRED FOR REVIEW, with concise reasons.

Finish with a short dad joke related to business or offices.
""")


### 💾 Save Agent Output to Table

This cell saves the agent's underwriting decision into the `agent_review` table in Unity Catalog.

- **Steps:**
  1. Creates a single-row DataFrame with `quote_id` and `agent_output`
  2. Registers it as a temporary view (`new_review_data`)
  3. Executes a `MERGE` to upsert the result into `lrcatalog.agentic_underwriting.agent_review`

- **Use case:** Stores decisions and justifications made by the agent for audit, governance, or follow-up review

In [0]:
#save output into a table
from pyspark.sql import Row

# Create a Spark DataFrame with new output
row = Row(quote_id=quote_id, agent_output=agent_output)
new_data = spark.createDataFrame([row])

# Register as temp view for merge
new_data.createOrReplaceTempView("new_review_data")

# Run MERGE to update or insert
spark.sql(f"""
MERGE INTO {catalog}.{schema}.global_agent_output AS target
USING new_review_data AS source
ON target.quote_id = source.quote_id
WHEN MATCHED THEN UPDATE SET target.agent_output = source.agent_output
WHEN NOT MATCHED THEN INSERT (quote_id, agent_output) VALUES (source.quote_id, source.agent_output)
""")

In [0]:
import json

result_summary = {
    "status": "success",
    "quote_id": quote_id,
    "catalog": catalog,
    "schema": schema,
    "table": "underwriting_results"
}

dbutils.notebook.exit(json.dumps(result_summary))